In [1]:
import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    hamming_loss,
    confusion_matrix,
)

In [2]:
current = Path.cwd()

PROJECT_ROOT = None

for path in [current] + list(current.parents):

    if (path / "src").is_dir():

        PROJECT_ROOT = path
        break


if PROJECT_ROOT is None:

    raise FileNotFoundError(
        "Could not find project root containing 'src'."
    )


if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


print(
    "Project root:",
    PROJECT_ROOT
)

Project root: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service


In [3]:
SEED = 42


def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "PyTorch:",
    torch.__version__
)

print(
    "Device:",
    DEVICE
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )

PyTorch: 2.14.0+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA: 13.0


In [4]:
CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "ecg"
    / "ptbxl"
    / "ptbxl_cnn_bilstm_attention_100hz_record_zscore.pt"
)


if not CHECKPOINT_PATH.exists():

    raise FileNotFoundError(
        "Current BiLSTM checkpoint not found:\n"
        f"{CHECKPOINT_PATH}"
    )


print(
    "Checkpoint:",
    CHECKPOINT_PATH
)

Checkpoint: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\checkpoints\ecg\ptbxl\ptbxl_cnn_bilstm_attention_100hz_record_zscore.pt


In [5]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)


print(
    "Checkpoint keys:"
)

print(
    list(checkpoint.keys())
)

Checkpoint keys:
['model_state_dict', 'model_config', 'train_config', 'augmentation_config', 'preprocessing_version', 'preprocessing_config', 'signal_length', 'num_leads', 'diagnostic_targets', 'rhythm_targets', 'diagnostic_thresholds', 'rhythm_thresholds', 'best_epoch', 'best_validation_score', 'seed']


In [6]:
MODEL_CONFIG = checkpoint[
    "model_config"
]

PREPROCESSING_VERSION = checkpoint[
    "preprocessing_version"
]

SIGNAL_LENGTH = int(
    checkpoint[
        "signal_length"
    ]
)

NUM_LEADS = int(
    checkpoint[
        "num_leads"
    ]
)

DIAGNOSTIC_COLUMNS = list(
    checkpoint[
        "diagnostic_targets"
    ]
)

RHYTHM_COLUMNS = list(
    checkpoint[
        "rhythm_targets"
    ]
)

DIAGNOSTIC_THRESHOLDS = np.asarray(
    checkpoint[
        "diagnostic_thresholds"
    ],
    dtype=np.float32,
)

RHYTHM_THRESHOLDS = np.asarray(
    checkpoint[
        "rhythm_thresholds"
    ],
    dtype=np.float32,
)


print(
    "Model configuration:"
)

print(
    json.dumps(
        MODEL_CONFIG,
        indent=2
    )
)


print(
    "\nPreprocessing:",
    PREPROCESSING_VERSION
)

print(
    "Signal length:",
    SIGNAL_LENGTH
)

print(
    "Number of leads:",
    NUM_LEADS
)

print(
    "Diagnostic labels:",
    DIAGNOSTIC_COLUMNS
)

print(
    "Rhythm labels:",
    RHYTHM_COLUMNS
)

Model configuration:
{
  "name": "cnn_bilstm_attention",
  "cnn_channels": 128,
  "lstm_hidden": 128,
  "lstm_layers": 2,
  "bidirectional": true,
  "dropout": 0.3,
  "cnn_kernel_1": 7,
  "cnn_kernel_2": 5
}

Preprocessing: 100hz_record_zscore
Signal length: 1000
Number of leads: 12
Diagnostic labels: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
Rhythm labels: ['RHYTHM_SR', 'RHYTHM_AFIB', 'RHYTHM_AFLT', 'RHYTHM_STACH', 'RHYTHM_SBRAD', 'RHYTHM_SARRH', 'RHYTHM_PSVT', 'RHYTHM_BIGU', 'RHYTHM_PACE']


In [7]:
diagnostic_threshold_table = pd.DataFrame({
    "label": DIAGNOSTIC_COLUMNS,
    "threshold": DIAGNOSTIC_THRESHOLDS,
})


rhythm_threshold_table = pd.DataFrame({
    "label": RHYTHM_COLUMNS,
    "threshold": RHYTHM_THRESHOLDS,
})


print(
    "Diagnostic thresholds:"
)

display(
    diagnostic_threshold_table
)


print(
    "Rhythm thresholds:"
)

display(
    rhythm_threshold_table
)

Diagnostic thresholds:


,label,threshold
0,NORM,0.78
1,MI,0.42
2,STTC,0.37
3,CD,0.44
4,HYP,0.50


Rhythm thresholds:


,label,threshold
0,RHYTHM_SR,0.41
1,RHYTHM_AFIB,0.77
2,RHYTHM_AFLT,0.95
3,RHYTHM_STACH,0.53
4,RHYTHM_SBRAD,0.68
5,RHYTHM_SARRH,0.27
6,RHYTHM_PSVT,0.44
7,RHYTHM_BIGU,0.33
8,RHYTHM_PACE,0.59


In [8]:
PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ecg"
    / "ptbxl"
    / PREPROCESSING_VERSION
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "manifests"
    / "ptbxl_manifest.csv"
)


if not PROCESSED_DIR.exists():

    raise FileNotFoundError(
        f"Processed directory not found:\n"
        f"{PROCESSED_DIR}"
    )


if not MANIFEST_PATH.exists():

    raise FileNotFoundError(
        f"PTB-XL manifest not found:\n"
        f"{MANIFEST_PATH}"
    )


print(
    "Processed directory:",
    PROCESSED_DIR
)

print(
    "Manifest:",
    MANIFEST_PATH
)

Processed directory: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore
Manifest: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\manifests\ptbxl_manifest.csv


In [9]:
ptbxl = pd.read_csv(
    MANIFEST_PATH
)


print(
    "Manifest shape:",
    ptbxl.shape
)


required_columns = [
    "ecg_id",
    "patient_id",
    "processed_path",
    "split",
]


missing_columns = [
    column
    for column in required_columns
    if column not in ptbxl.columns
]


if missing_columns:

    raise ValueError(
        "Missing required manifest columns:\n"
        f"{missing_columns}"
    )


all_label_columns = (
    DIAGNOSTIC_COLUMNS
    + RHYTHM_COLUMNS
)


missing_labels = [
    column
    for column in all_label_columns
    if column not in ptbxl.columns
]


if missing_labels:

    raise ValueError(
        "Missing target columns:\n"
        f"{missing_labels}"
    )


print(
    "\nColumns verified."
)

print(
    "Diagnostic:",
    DIAGNOSTIC_COLUMNS
)

print(
    "Rhythm:",
    RHYTHM_COLUMNS
)

Manifest shape: (21837, 25)

Columns verified.
Diagnostic: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
Rhythm: ['RHYTHM_SR', 'RHYTHM_AFIB', 'RHYTHM_AFLT', 'RHYTHM_STACH', 'RHYTHM_SBRAD', 'RHYTHM_SARRH', 'RHYTHM_PSVT', 'RHYTHM_BIGU', 'RHYTHM_PACE']


In [10]:
train_df = (
    ptbxl[
        ptbxl["split"] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)


val_df = (
    ptbxl[
        ptbxl["split"] == "val"
    ]
    .copy()
    .reset_index(drop=True)
)


test_df = (
    ptbxl[
        ptbxl["split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    f"Train      : {len(train_df):,}"
)

print(
    f"Validation : {len(val_df):,}"
)

print(
    f"Test       : {len(test_df):,}"
)

Train      : 17,441
Validation : 2,193
Test       : 2,203


In [11]:
train_patients = set(
    train_df["patient_id"]
)

val_patients = set(
    val_df["patient_id"]
)

test_patients = set(
    test_df["patient_id"]
)


train_val_overlap = (
    train_patients
    & val_patients
)

train_test_overlap = (
    train_patients
    & test_patients
)

val_test_overlap = (
    val_patients
    & test_patients
)


print(
    "Train ∩ Validation:",
    len(train_val_overlap)
)

print(
    "Train ∩ Test:",
    len(train_test_overlap)
)

print(
    "Validation ∩ Test:",
    len(val_test_overlap)
)


if (
    len(train_val_overlap) > 0
    or
    len(train_test_overlap) > 0
    or
    len(val_test_overlap) > 0
):

    raise RuntimeError(
        "Patient leakage detected."
    )


print(
    "\nPatient-level split integrity PASSED."
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

Patient-level split integrity PASSED.


In [12]:
from src.datasets.ecg_dataset import (
    PTBXLDataset
)


test_dataset = PTBXLDataset(

    signals=None,

    labels_df=test_df,

    superclass_cols=
        DIAGNOSTIC_COLUMNS,

    rhythm_cols=
        RHYTHM_COLUMNS,

    record_paths=
        test_df[
            "processed_path"
        ].tolist(),

    project_root=
        PROJECT_ROOT,

    signal_length=
        SIGNAL_LENGTH,

    num_leads=
        NUM_LEADS,
)


print(
    "Test dataset size:",
    len(test_dataset)
)

Test dataset size: 2203


In [13]:
sample = test_dataset[0]


print(
    "Signal shape:",
    sample["signal"].shape
)

print(
    "Diagnostic shape:",
    sample["diagnostic"].shape
)

print(
    "Rhythm shape:",
    sample["rhythm"].shape
)

print(
    "Record:",
    sample["record_path"]
)

Signal shape: torch.Size([12, 1000])
Diagnostic shape: torch.Size([5])
Rhythm shape: torch.Size([9])
Record: data/processed/ecg/ptbxl/100hz_record_zscore/waveforms/00000009.npy


In [14]:
TEST_BATCH_SIZE = 32


test_loader = DataLoader(

    test_dataset,

    batch_size=
        TEST_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available(),
)


print(
    "Test batches:",
    len(test_loader)
)

Test batches: 69


In [15]:
from src.models.ecg.cnn_bilstm_attention import (
    CNNBiLSTMAttention
)


model = CNNBiLSTMAttention(

    num_leads=
        NUM_LEADS,

    num_diagnostic_classes=
        len(DIAGNOSTIC_COLUMNS),

    num_rhythm_classes=
        len(RHYTHM_COLUMNS),

    cnn_channels=
        MODEL_CONFIG[
            "cnn_channels"
        ],

    lstm_hidden=
        MODEL_CONFIG[
            "lstm_hidden"
        ],

    lstm_layers=
        MODEL_CONFIG[
            "lstm_layers"
        ],

    dropout=
        MODEL_CONFIG[
            "dropout"
        ],
)


model = model.to(
    DEVICE
)


print(
    model
)

CNNBiLSTMAttention(
  (cnn1): ConvNormPool(
    (conv1): Conv1d(12, 128, kernel_size=(7,), stride=(1,), padding=(3,))
    (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv2): Conv1d(128, 128, kernel_size=(7,), stride=(1,), padding=(3,))
    (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (activation): Swish()
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (cnn2): ConvNormPool(
    (conv1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv2): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (activation): Swish()
    (pool): MaxPool1d(kernel_size

In [16]:
state_dict = checkpoint[
    "model_state_dict"
]


clean_state_dict = {}


for key, value in state_dict.items():

    new_key = key

    if new_key.startswith(
        "module."
    ):

        new_key = new_key[
            len("module.") :
        ]


    clean_state_dict[
        new_key
    ] = value


missing_keys, unexpected_keys = (
    model.load_state_dict(
        clean_state_dict,
        strict=False,
    )
)


print(
    "Missing keys:",
    missing_keys
)

print(
    "Unexpected keys:",
    unexpected_keys
)


if (
    len(missing_keys) != 0
    or
    len(unexpected_keys) != 0
):

    raise RuntimeError(
        "Checkpoint/model mismatch."
    )


model.eval()


print(
    "\nCurrent BiLSTM checkpoint loaded successfully."
)

Missing keys: []
Unexpected keys: []

Current BiLSTM checkpoint loaded successfully.


In [17]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)


trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

Total parameters: 1,018,255
Trainable parameters: 1,018,255


In [18]:
@torch.no_grad()
def collect_predictions(
    model,
    loader,
):

    model.eval()


    diagnostic_true = []
    diagnostic_probs = []

    rhythm_true = []
    rhythm_probs = []


    for batch in loader:

        signals = batch[
            "signal"
        ].to(
            DEVICE,
            non_blocking=True,
        )


        diagnostic_targets = (
            batch[
                "diagnostic"
            ]
            .to(
                DEVICE,
                non_blocking=True,
            )
        )


        rhythm_targets = (
            batch[
                "rhythm"
            ]
            .to(
                DEVICE,
                non_blocking=True,
            )
        )


        outputs = model(
            signals
        )


        diagnostic_true.append(
            diagnostic_targets
            .cpu()
            .numpy()
        )


        diagnostic_probs.append(
            torch.sigmoid(
                outputs[
                    "diagnostic"
                ]
            )
            .cpu()
            .numpy()
        )


        rhythm_true.append(
            rhythm_targets
            .cpu()
            .numpy()
        )


        rhythm_probs.append(
            torch.sigmoid(
                outputs[
                    "rhythm"
                ]
            )
            .cpu()
            .numpy()
        )


    diagnostic_true = np.concatenate(
        diagnostic_true,
        axis=0,
    )


    diagnostic_probs = np.concatenate(
        diagnostic_probs,
        axis=0,
    )


    rhythm_true = np.concatenate(
        rhythm_true,
        axis=0,
    )


    rhythm_probs = np.concatenate(
        rhythm_probs,
        axis=0,
    )


    return {
        "diagnostic_true":
            diagnostic_true,

        "diagnostic_probs":
            diagnostic_probs,

        "rhythm_true":
            rhythm_true,

        "rhythm_probs":
            rhythm_probs,
    }


predictions = collect_predictions(
    model,
    test_loader,
)


diagnostic_true = predictions[
    "diagnostic_true"
]

diagnostic_probs = predictions[
    "diagnostic_probs"
]

rhythm_true = predictions[
    "rhythm_true"
]

rhythm_probs = predictions[
    "rhythm_probs"
]


print(
    "Diagnostic:",
    diagnostic_true.shape
)

print(
    "Rhythm:",
    rhythm_true.shape
)

print(
    "Test records:",
    len(diagnostic_true)
)

Diagnostic: (2203, 5)
Rhythm: (2203, 9)
Test records: 2203


In [19]:
diagnostic_pred = (
    diagnostic_probs
    >= DIAGNOSTIC_THRESHOLDS
).astype(int)


rhythm_pred = (
    rhythm_probs
    >= RHYTHM_THRESHOLDS
).astype(int)


print(
    "Diagnostic predictions:",
    diagnostic_pred.shape
)

print(
    "Rhythm predictions:",
    rhythm_pred.shape
)

Diagnostic predictions: (2203, 5)
Rhythm predictions: (2203, 9)


In [20]:
def safe_macro_auroc(
    y_true,
    y_prob,
):

    scores = []


    for i in range(
        y_true.shape[1]
    ):

        if len(
            np.unique(
                y_true[:, i]
            )
        ) < 2:

            continue


        scores.append(
            roc_auc_score(
                y_true[:, i],
                y_prob[:, i],
            )
        )


    if not scores:

        return float("nan")


    return float(
        np.mean(scores)
    )


def safe_macro_auprc(
    y_true,
    y_prob,
):

    scores = []


    for i in range(
        y_true.shape[1]
    ):

        if len(
            np.unique(
                y_true[:, i]
            )
        ) < 2:

            continue


        scores.append(
            average_precision_score(
                y_true[:, i],
                y_prob[:, i],
            )
        )


    if not scores:

        return float("nan")


    return float(
        np.mean(scores)
    )


def calculate_metrics(
    y_true,
    y_prob,
    y_pred,
):

    return {

        "AUROC":
            safe_macro_auroc(
                y_true,
                y_prob,
            ),

        "AUPRC":
            safe_macro_auprc(
                y_true,
                y_prob,
            ),

        "Macro F1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "Macro Precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "Macro Recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "Micro F1":
            f1_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0,
            ),

        # Exact multilabel match:
        "Exact Match Accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        # 1 - fraction of incorrectly predicted label entries:
        "Hamming Accuracy":
            1.0
            -
            hamming_loss(
                y_true,
                y_pred,
            ),
    }

In [21]:
diagnostic_metrics = calculate_metrics(

    diagnostic_true,

    diagnostic_probs,

    diagnostic_pred,
)


print(
    "=" * 75
)

print(
    "FINAL DIAGNOSTIC TEST RESULTS"
)

print(
    "=" * 75
)


for key, value in diagnostic_metrics.items():

    print(
        f"{key:<25}: "
        f"{value:.6f}"
    )

FINAL DIAGNOSTIC TEST RESULTS
AUROC                    : 0.925416
AUPRC                    : 0.815309
Macro F1                 : 0.742128
Macro Precision          : 0.751426
Macro Recall             : 0.738740
Micro F1                 : 0.773941
Exact Match Accuracy     : 0.620064
Hamming Accuracy         : 0.884703


In [22]:
rhythm_metrics = calculate_metrics(

    rhythm_true,

    rhythm_probs,

    rhythm_pred,
)


print(
    "=" * 75
)

print(
    "FINAL RHYTHM TEST RESULTS"
)

print(
    "=" * 75
)


for key, value in rhythm_metrics.items():

    print(
        f"{key:<25}: "
        f"{value:.6f}"
    )

FINAL RHYTHM TEST RESULTS
AUROC                    : 0.921660
AUPRC                    : 0.583981
Macro F1                 : 0.550995
Macro Precision          : 0.598687
Macro Recall             : 0.637930
Micro F1                 : 0.849799
Exact Match Accuracy     : 0.774399
Hamming Accuracy         : 0.966107


In [23]:
comparison_df = pd.DataFrame(
    [
        {
            "Task":
                "Diagnostic",

            "AUROC":
                diagnostic_metrics[
                    "AUROC"
                ],

            "AUPRC":
                diagnostic_metrics[
                    "AUPRC"
                ],

            "Macro F1":
                diagnostic_metrics[
                    "Macro F1"
                ],

            "Precision":
                diagnostic_metrics[
                    "Macro Precision"
                ],

            "Recall":
                diagnostic_metrics[
                    "Macro Recall"
                ],

            "Exact Match Accuracy":
                diagnostic_metrics[
                    "Exact Match Accuracy"
                ],

            "Hamming Accuracy":
                diagnostic_metrics[
                    "Hamming Accuracy"
                ],
        },

        {
            "Task":
                "Rhythm",

            "AUROC":
                rhythm_metrics[
                    "AUROC"
                ],

            "AUPRC":
                rhythm_metrics[
                    "AUPRC"
                ],

            "Macro F1":
                rhythm_metrics[
                    "Macro F1"
                ],

            "Precision":
                rhythm_metrics[
                    "Macro Precision"
                ],

            "Recall":
                rhythm_metrics[
                    "Macro Recall"
                ],

            "Exact Match Accuracy":
                rhythm_metrics[
                    "Exact Match Accuracy"
                ],

            "Hamming Accuracy":
                rhythm_metrics[
                    "Hamming Accuracy"
                ],
        },
    ]
)


display(
    comparison_df
)

,Task,AUROC,AUPRC,Macro F1,Precision,Recall,Exact Match Accuracy,Hamming Accuracy
0,Diagnostic,0.925416,0.815309,0.742128,0.751426,0.73874,0.620064,0.884703
1,Rhythm,0.921660,0.583981,0.550995,0.598687,0.63793,0.774399,0.966107


In [24]:
diagnostic_rows = []


for i, label in enumerate(
    DIAGNOSTIC_COLUMNS
):

    true = diagnostic_true[:, i]

    prob = diagnostic_probs[:, i]

    pred = diagnostic_pred[:, i]


    if len(
        np.unique(true)
    ) >= 2:

        auroc = roc_auc_score(
            true,
            prob
        )

        auprc = average_precision_score(
            true,
            prob
        )

    else:

        auroc = np.nan

        auprc = np.nan


    diagnostic_rows.append({

        "Label":
            label,

        "AUROC":
            auroc,

        "AUPRC":
            auprc,

        "F1":
            f1_score(
                true,
                pred,
                zero_division=0,
            ),

        "Precision":
            precision_score(
                true,
                pred,
                zero_division=0,
            ),

        "Recall":
            recall_score(
                true,
                pred,
                zero_division=0,
            ),

        "Accuracy":
            np.mean(
                true == pred
            ),

        "TP":
            confusion_matrix(
                true,
                pred,
                labels=[0, 1],
            )[1, 1],

        "FP":
            confusion_matrix(
                true,
                pred,
                labels=[0, 1],
            )[0, 1],

        "FN":
            confusion_matrix(
                true,
                pred,
                labels=[0, 1],
            )[1, 0],

        "TN":
            confusion_matrix(
                true,
                pred,
                labels=[0, 1],
            )[0, 0],
    })


diagnostic_report = pd.DataFrame(
    diagnostic_rows
)


display(
    diagnostic_report
)

,Label,AUROC,AUPRC,F1,Precision,Recall,Accuracy,TP,FP,FN,TN
0,NORM,0.948165,0.925640,0.859113,0.818011,0.904564,0.870177,872,194,92,1045
1,MI,0.919427,0.822991,0.716730,0.755511,0.681736,0.864730,377,122,176,1528
2,STTC,0.929271,0.807906,0.763276,0.721088,0.810707,0.880617,424,164,99,1516
3,CD,0.920459,0.840215,0.750831,0.837037,0.680723,0.897867,339,66,159,1639
4,HYP,0.909759,0.679793,0.620690,0.625483,0.615970,0.910123,162,97,101,1843


In [25]:
rhythm_rows = []


for i, label in enumerate(
    RHYTHM_COLUMNS
):

    true = rhythm_true[:, i]

    prob = rhythm_probs[:, i]

    pred = rhythm_pred[:, i]


    if len(
        np.unique(true)
    ) >= 2:

        auroc = roc_auc_score(
            true,
            prob
        )

        auprc = average_precision_score(
            true,
            prob
        )

    else:

        auroc = np.nan

        auprc = np.nan


    rhythm_rows.append({

        "Label":
            label,

        "AUROC":
            auroc,

        "AUPRC":
            auprc,

        "F1":
            f1_score(
                true,
                pred,
                zero_division=0,
            ),

        "Precision":
            precision_score(
                true,
                pred,
                zero_division=0,
            ),

        "Recall":
            recall_score(
                true,
                pred,
                zero_division=0,
            ),

        "Accuracy":
            np.mean(
                true == pred
            ),
    })


rhythm_report = pd.DataFrame(
    rhythm_rows
)


display(
    rhythm_report
)

,Label,AUROC,AUPRC,F1,Precision,Recall,Accuracy
0,RHYTHM_SR,0.861690,0.936209,0.924000,0.887486,0.963647,0.879256
1,RHYTHM_AFIB,0.980100,0.867873,0.835526,0.835526,0.835526,0.977304
2,RHYTHM_AFLT,0.953227,0.599440,0.444444,1.000000,0.285714,0.997730
3,RHYTHM_STACH,0.991761,0.880617,0.855556,0.785714,0.939024,0.988198
4,RHYTHM_SBRAD,0.949692,0.523760,0.491525,0.537037,0.453125,0.972764
5,RHYTHM_SARRH,0.723192,0.113451,0.181159,0.125628,0.324675,0.897413
6,RHYTHM_PSVT,0.998637,0.325000,0.222222,0.125000,1.000000,0.993645
7,RHYTHM_BIGU,0.880752,0.223763,0.235294,0.222222,0.250000,0.994099
8,RHYTHM_PACE,0.955889,0.785712,0.769231,0.869565,0.689655,0.994553


In [26]:
for i, label in enumerate(
    DIAGNOSTIC_COLUMNS
):

    cm = confusion_matrix(
        diagnostic_true[:, i],
        diagnostic_pred[:, i],
        labels=[0, 1],
    )


    print(
        f"\n{label}"
    )

    print(
        "TN:",
        cm[0, 0],
        " FP:",
        cm[0, 1]
    )

    print(
        "FN:",
        cm[1, 0],
        " TP:",
        cm[1, 1]
    )


NORM
TN: 1045  FP: 194
FN: 92  TP: 872

MI
TN: 1528  FP: 122
FN: 176  TP: 377

STTC
TN: 1516  FP: 164
FN: 99  TP: 424

CD
TN: 1639  FP: 66
FN: 159  TP: 339

HYP
TN: 1843  FP: 97
FN: 101  TP: 162


In [27]:
prediction_rows = []


for i in range(
    len(diagnostic_true)
):

    row = {

        "record_path":
            test_df.iloc[i][
                "processed_path"
            ],
    }


    for j, label in enumerate(
        DIAGNOSTIC_COLUMNS
    ):

        row[
            f"{label}_true"
        ] = diagnostic_true[i, j]

        row[
            f"{label}_prob"
        ] = diagnostic_probs[i, j]

        row[
            f"{label}_pred"
        ] = diagnostic_pred[i, j]


    for j, label in enumerate(
        RHYTHM_COLUMNS
    ):

        row[
            f"{label}_true"
        ] = rhythm_true[i, j]

        row[
            f"{label}_prob"
        ] = rhythm_probs[i, j]

        row[
            f"{label}_pred"
        ] = rhythm_pred[i, j]


    prediction_rows.append(
        row
    )


predictions_df = pd.DataFrame(
    prediction_rows
)


RESULTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "metrics"
    / "ecg"
    / "cnn_bilstm_attention"
)


RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


predictions_path = (
    RESULTS_DIR
    / "ptbxl_test_predictions.csv"
)


predictions_df.to_csv(
    predictions_path,
    index=False
)


print(
    "Saved:",
    predictions_path
)

Saved: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\metrics\ecg\cnn_bilstm_attention\ptbxl_test_predictions.csv


In [28]:
summary_path = (
    RESULTS_DIR
    / "ptbxl_bilstm_evaluation.csv"
)


summary_df = pd.DataFrame(
    [
        {
            "Task":
                "Diagnostic",
            **diagnostic_metrics
        },

        {
            "Task":
                "Rhythm",
            **rhythm_metrics
        },
    ]
)


summary_df.to_csv(
    summary_path,
    index=False
)


display(
    summary_df
)


print(
    "\nSaved:",
    summary_path
)

,Task,AUROC,AUPRC,Macro F1,Macro Precision,Macro Recall,Micro F1,Exact Match Accuracy,Hamming Accuracy
0,Diagnostic,0.925416,0.815309,0.742128,0.751426,0.73874,0.773941,0.620064,0.884703
1,Rhythm,0.921660,0.583981,0.550995,0.598687,0.63793,0.849799,0.774399,0.966107



Saved: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\metrics\ecg\cnn_bilstm_attention\ptbxl_bilstm_evaluation.csv


In [29]:
checks = {

    "Checkpoint exists":
        CHECKPOINT_PATH.exists(),

    "Manifest exists":
        MANIFEST_PATH.exists(),

    "12 leads":
        NUM_LEADS == 12,

    "1000 samples":
        SIGNAL_LENGTH == 1000,

    "5 diagnostic labels":
        len(DIAGNOSTIC_COLUMNS) == 5,

    "9 rhythm labels":
        len(RHYTHM_COLUMNS) == 9,

    "No train/val patient overlap":
        len(train_val_overlap) == 0,

    "No train/test patient overlap":
        len(train_test_overlap) == 0,

    "No val/test patient overlap":
        len(val_test_overlap) == 0,

    "All test records predicted":
        len(diagnostic_true)
        == len(test_dataset),

    "Diagnostic thresholds loaded":
        len(DIAGNOSTIC_THRESHOLDS)
        == len(DIAGNOSTIC_COLUMNS),

    "Rhythm thresholds loaded":
        len(RHYTHM_THRESHOLDS)
        == len(RHYTHM_COLUMNS),
}


for name, passed in checks.items():

    print(
        "[PASS]"
        if passed
        else "[FAIL]",
        name
    )


if not all(
    checks.values()
):

    raise RuntimeError(
        "One or more evaluation integrity checks failed."
    )


print(
    "\nAll evaluation checks PASSED."
)

[PASS] Checkpoint exists
[PASS] Manifest exists
[PASS] 12 leads
[PASS] 1000 samples
[PASS] 5 diagnostic labels
[PASS] 9 rhythm labels
[PASS] No train/val patient overlap
[PASS] No train/test patient overlap
[PASS] No val/test patient overlap
[PASS] All test records predicted
[PASS] Diagnostic thresholds loaded
[PASS] Rhythm thresholds loaded

All evaluation checks PASSED.
